# Staging â€” DuckDB Native

Replaces the pandas row-by-row CSV reader with DuckDB's `read_csv` which is
orders of magnitude faster.

- **Resumable** â€” skips files already staged (checks for existing parquet)
- **No destructive reset** â€” never wipes the stage folder
- **Controllable** â€” set `MAX_FILES` to limit how many files are processed

In [7]:
## Config

import duckdb
import json
import os
from pathlib import Path
import time

RAW_DIR   = Path("/home/jovyan/data/sim/raw/sim_test_decoded/2025/")
STAGE_DIR = Path("/home/jovyan/data/stage/handover_events")
# Use native Linux filesystem for DuckDB temp spill â€” mmap works here,
# unlike the Windows NTFS mount at /home/jovyan/data/
TMP_DIR   = "/tmp/duckdb_tmp"

# Column positions (zero-based) in the raw CSV
# id, test_type, creation_time, collection_time, insert_time, sim_id, ...
COL_EVENT_TS   = '03'   # collection_time
COL_VEHICLE_ID = '05'   # sim_id
COL_IMSI       = '13' # imsi
COL_CELL_ID    = '18' # cell_id
COL_RAT        = '23' # technology (4G/5G)
COL_PCI1_RSRP  = '34'  # pci_1_rsrp  — neighbour cell RSRP
COL_PCI1_RSRQ  = '35'  # pci_1_rsrq  — neighbour cell RSRQ
COL_PING1      = '84'  # ping1
COL_PING2      = '85'  # ping2
COL_PING3      = '86'  # ping3
COL_PING4      = '87'  # ping4


# Set to an integer to limit files processed (useful for testing)
# Set to None to process all files
MAX_FILES = None

STAGE_DIR.mkdir(parents=True, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

print(f"RAW_DIR:   {RAW_DIR}")
print(f"STAGE_DIR: {STAGE_DIR}")
print(f"TMP_DIR:   {TMP_DIR}")
print(f"MAX_FILES: {MAX_FILES}")

RAW_DIR:   /home/jovyan/data/sim/raw/sim_test_decoded/2025
STAGE_DIR: /home/jovyan/data/stage/handover_events
TMP_DIR:   /tmp/duckdb_tmp
MAX_FILES: None


In [8]:
## Connect DuckDB

import os, psutil

TOTAL_RAM_GB = round(psutil.virtual_memory().total / (1024**3), 1)
DUCKDB_THREADS = max(1, min(4, os.cpu_count() or 4))
# Use 75% of available RAM â€” safe headroom for OS + Jupyter kernel overhead
DUCKDB_MEM_GB = max(4, int(TOTAL_RAM_GB * 0.75))

con = duckdb.connect()  # in-memory â€” staging doesn't need persistence
con.execute(f"SET temp_directory='{TMP_DIR}'")
con.execute(f"SET memory_limit='{DUCKDB_MEM_GB}GB'")
con.execute(f"SET threads={DUCKDB_THREADS}")
con.execute("SET preserve_insertion_order=false")

print(f"CPUs: {os.cpu_count()}")
print(f"RAM:  {TOTAL_RAM_GB} GB")
print(f"DuckDB memory_limit: {DUCKDB_MEM_GB}GB")
print(f"DuckDB threads: {DUCKDB_THREADS}")

CPUs: 4
RAM:  23.5 GB
DuckDB memory_limit: 17GB
DuckDB threads: 4


In [9]:
## Find raw files

exts = {".gz", ".csv", ".txt"}
all_files = sorted([
    p for p in RAW_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in exts
])

raw_files = all_files[:MAX_FILES] if MAX_FILES else all_files

print(f"Total files found: {len(all_files)}")
print(f"Files to consider: {len(raw_files)}")

Total files found: 364
Files to consider: 364


In [10]:
## Process files â€” resumable, DuckDB native

skipped = 0
processed = 0
total_good = 0
total_bad = 0
run_start = time.time()

for i, path in enumerate(raw_files, 1):

    # Derive the expected output path from the filename date
    # e.g. 2025_06_15.txt -> event_date=2025-06-15
    stem = path.stem  # e.g. '2025_06_15'
    event_date = stem.replace('_', '-')  # e.g. '2025-06-15'
    out_dir  = STAGE_DIR / f"event_date={event_date}"
    out_file = out_dir / "stage.parquet"

    # --- Resumable: skip if already staged ---
    if out_file.exists():
        skipped += 1
        print(f"[{i}/{len(raw_files)}] SKIP {path.name}")
        continue

    file_start = time.time()

    try:
        out_dir.mkdir(parents=True, exist_ok=True)

        # DuckDB reads the CSV and writes parquet in one shot
        # column0..columnN maps to zero-based column positions
        result = con.execute(f"""
            COPY (
                SELECT
                    column{COL_VEHICLE_ID}  AS vehicle_id,
                    column{COL_IMSI}        AS imsi,
                    column{COL_EVENT_TS}    AS event_ts,
                    column{COL_CELL_ID}     AS cell_id,
                    column{COL_RAT}         AS rat,
                    '{path.name}'           AS source_file,
                    TRY_CAST(column{COL_PCI1_RSRP} AS DOUBLE) AS pci_1_rsrp,
                    TRY_CAST(column{COL_PCI1_RSRQ} AS DOUBLE) AS pci_1_rsrq,
                    TRY_CAST(column{COL_PING1}     AS DOUBLE) AS ping1,
                    TRY_CAST(column{COL_PING2}     AS DOUBLE) AS ping2,
                    TRY_CAST(column{COL_PING3}     AS DOUBLE) AS ping3,
                    TRY_CAST(column{COL_PING4}     AS DOUBLE) AS ping4
                FROM read_csv(
                    '{path}',
                    header        = false,
                    ignore_errors = true,
                    compression   = 'auto',
                    delim         = ','
                )
                WHERE column{COL_VEHICLE_ID} IS NOT NULL
                  AND column{COL_CELL_ID}    IS NOT NULL
                  AND column{COL_EVENT_TS}   IS NOT NULL
            ) TO '{out_file}' (FORMAT PARQUET)
        """)

        elapsed = time.time() - file_start
        # Count rows written
        n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_file}')").fetchone()[0]
        total_good += n_rows
        processed += 1
        pct = 100.0 * i / len(raw_files)
        run_elapsed = time.time() - run_start
        rate = processed / run_elapsed  # files/sec
        remaining = len(raw_files) - i
        eta_s = remaining / rate if rate > 0 else 0
        _h, _rem = divmod(int(eta_s), 3600); _m, _s = divmod(_rem, 60)
        eta_str = f"{_h}h{_m:02d}m" if _h > 0 else f"{_m}m{_s:02d}s"
        print(
            f"[{i}/{len(raw_files)}] {path.name} → {n_rows:,} rows "
            f"({elapsed:.1f}s) | {pct:.0f}% | ETA {eta_str}",
            flush=True,
        )
        with open("/tmp/stage_progress.json", "w") as _pf:
            json.dump({
                "processed": processed, "skipped": skipped, "errors": total_bad,
                "total_files": len(raw_files), "pct_done": round(pct, 1),
                "total_rows": total_good,
                "elapsed_min": round(run_elapsed / 60, 1),
                "eta_min": round(eta_s / 60, 1),
                "last_file": path.name,
                "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            }, _pf, indent=2)

    except Exception as e:
        total_bad += 1
        print(f"[{i}/{len(raw_files)}] ERROR {path.name}: {e}")

total_elapsed = time.time() - run_start
print(f"\nDone. {processed} processed, {skipped} skipped, {total_bad} errors")
print(f"Total rows staged: {total_good:,}")
print(f"Total runtime: {total_elapsed/60:.1f} min")

[1/364] SKIP 2025_01_01.gz
[2/364] SKIP 2025_01_02.gz
[3/364] SKIP 2025_01_03.gz
[4/364] SKIP 2025_01_04.gz
[5/364] SKIP 2025_01_05.gz
[6/364] SKIP 2025_01_06.gz
[7/364] SKIP 2025_01_07.gz
[8/364] SKIP 2025_01_08.gz
[9/364] SKIP 2025_01_09.gz
[10/364] SKIP 2025_01_10.gz
[11/364] SKIP 2025_01_11.gz
[12/364] SKIP 2025_01_12.gz
[13/364] SKIP 2025_01_13.gz
[14/364] SKIP 2025_01_14.gz
[15/364] SKIP 2025_01_15.gz
[16/364] SKIP 2025_01_17.gz
[17/364] SKIP 2025_01_18.gz
[18/364] SKIP 2025_01_19.gz
[19/364] SKIP 2025_01_20.gz
[20/364] SKIP 2025_01_21.gz
[21/364] SKIP 2025_01_22.gz
[22/364] SKIP 2025_01_23.gz
[23/364] SKIP 2025_01_24.gz
[24/364] SKIP 2025_01_25.gz
[25/364] SKIP 2025_01_26.gz
[26/364] SKIP 2025_01_27.gz
[27/364] SKIP 2025_01_28.gz
[28/364] SKIP 2025_01_29.gz
[29/364] SKIP 2025_01_30.gz
[30/364] SKIP 2025_01_31.gz
[31/364] SKIP 2025_02_01.gz
[32/364] SKIP 2025_02_02.gz
[33/364] SKIP 2025_02_03.gz
[34/364] SKIP 2025_02_04.gz
[35/364] SKIP 2025_02_05.gz
[36/364] SKIP 2025_02_06.gz
[

In [11]:
## Verify â€” check a sample parquet file

sample_files = list(STAGE_DIR.rglob("*.parquet"))
print(f"Parquet files on disk: {len(sample_files)}")

con.execute(f"""
    SELECT *
    FROM read_parquet('{sample_files[0]}')
    LIMIT 3
""").df()

Parquet files on disk: 478


,vehicle_id,imsi,event_ts,cell_id,rat,source_file,event_date
0,F140A413CDBACB245F79DAE1D7438EC4,000000000000000,2025-01-01 00:02:01.793,69492489,4G,2025_01_01.gz,2025-01-01
1,F140A413CDBACB245F79DAE1D7438EC4,000000000000000,2025-01-01 00:10:05.626,69492489,4G,2025_01_01.gz,2025-01-01
2,F140A413CDBACB245F79DAE1D7438EC4,000000000000000,2025-01-01 00:18:09.436,69492708,4G,2025_01_01.gz,2025-01-01


In [ ]:
## Register handover_events in analytics DuckDB

import os
import psutil

# Set to True to force a full table rebuild even if it already exists.
FORCE_REBUILD = False

# DuckDB in this environment runs on CPU threads.
# GPU acceleration is not available in this notebook path.
USE_GPU = False

# Free memory from both connections
for _conn in ('con', 'analytics'):
    try:
        globals()[_conn].close()
    except Exception:
        pass

ANALYTICS_DB = "/home/jovyan/data/analytics.duckdb"

# Increase parallelism for faster initial rebuilds, capped to stay stable in container.
ANALYTICS_THREADS = max(2, min(8, os.cpu_count() or 4))

# Use currently available RAM (not total RAM) to avoid OOM in constrained containers.
available_gb = max(1, int(psutil.virtual_memory().available / (1024**3)))
ANALYTICS_MEM_GB = max(2, min(12, int(available_gb * 0.70)))


def connect_analytics(mem_gb: int):
    return duckdb.connect(
        ANALYTICS_DB,
        config={
            "temp_directory": TMP_DIR,
            "memory_limit": f"{mem_gb}GB",
            "threads": str(ANALYTICS_THREADS),
            "preserve_insertion_order": "false",
            "enable_progress_bar": "true",
        },
    )


try:
    analytics = connect_analytics(ANALYTICS_MEM_GB)
except Exception as e:
    retry_mem_gb = max(1, ANALYTICS_MEM_GB // 2)
    print(f"Connect failed at {ANALYTICS_MEM_GB}GB ({e}); retrying at {retry_mem_gb}GB...")
    analytics = connect_analytics(retry_mem_gb)
    ANALYTICS_MEM_GB = retry_mem_gb

print(f"Analytics DB: {ANALYTICS_DB}")
print(f"Available RAM now: {available_gb}GB")
print(f"Analytics memory_limit: {ANALYTICS_MEM_GB}GB")
print(f"Analytics threads: {ANALYTICS_THREADS}")
print(f"GPU enabled: {USE_GPU} (DuckDB path here is CPU-only)")

# --- Resumable: skip rebuild if table already exists and FORCE_REBUILD is False ---
_table_exists = analytics.execute(
    "SELECT COUNT(*) FROM information_schema.tables "
    "WHERE table_name = 'handover_events'"
).fetchone()[0] > 0

if _table_exists and not FORCE_REBUILD:
    print("handover_events already exists â€” skipping rebuild (set FORCE_REBUILD=True to override).")
else:
    day_dirs = sorted([p for p in STAGE_DIR.glob("event_date=*") if p.is_dir()])
    if not day_dirs:
        raise FileNotFoundError(f"No staged day partitions found under {STAGE_DIR}")

    if FORCE_REBUILD:
        print("FORCE_REBUILD=True â€” dropping and rebuilding handover_events...")
    else:
        print("handover_events not found â€” building for the first time...")

    analytics.execute("DROP TABLE IF EXISTS handover_events")
    build_start = time.time()
    total_days = len(day_dirs)
    print(f"Building from {total_days} daily partitions...")

    for idx, day_dir in enumerate(day_dirs, start=1):
        day_glob = str(day_dir / "*.parquet")
        day_name = day_dir.name.replace("event_date=", "")

        day_select = f"""
            SELECT
                vehicle_id,
                imsi,
                event_ts,
                cell_id,
                rat,
                source_file,
                CAST(event_ts AS DATE) AS event_date
            FROM read_parquet('{day_glob}')
            WHERE vehicle_id IS NOT NULL
              AND cell_id    IS NOT NULL
              AND event_ts   IS NOT NULL
        """

        if idx == 1:
            analytics.execute(f"CREATE TABLE handover_events AS {day_select}")
        else:
            analytics.execute(f"INSERT INTO handover_events {day_select}")

        if idx == 1 or idx % 10 == 0 or idx == total_days:
            elapsed_min = (time.time() - build_start) / 60
            print(f"[{idx}/{total_days}] loaded {day_name} | elapsed {elapsed_min:.1f} min")

    print(f"Build complete in {(time.time() - build_start)/60:.1f} min")

try:
    summary = analytics.execute("""
        SELECT
            COUNT(*)                                AS total_rows,
            APPROX_COUNT_DISTINCT(vehicle_id)       AS approx_vehicles,
            APPROX_COUNT_DISTINCT(cell_id)          AS approx_cells,
            MIN(event_ts)                           AS min_event,
            MAX(event_ts)                           AS max_event
        FROM handover_events
    """).df()
except duckdb.OutOfMemoryException:
    print("Summary query hit OOM; returning lightweight stats instead.")
    summary = analytics.execute("""
        SELECT
            COUNT(*)      AS total_rows,
            MIN(event_ts) AS min_event,
            MAX(event_ts) AS max_event
        FROM handover_events
    """).df()
summary

Analytics DB: /home/jovyan/data/analytics.duckdb
Available RAM now: 20GB
Analytics memory_limit: 8GB
Analytics threads: 1
handover_events not found â€” building for the first time...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,approx_vehicles,approx_cells,min_event,max_event
0,1735661691,87876,1197011,2025-01-01 00:02:01.793,2025-12-31 23:59:56.589
